# Proof of Concept — Notebook B (Team Blue)

Runs in its own kernel, separate from Notebook A. Does NOT start the
competition — Notebook A does. Joins as `team_blue` and plays.

**Critical:** `SERVER_IP` / `SERVER_PORT` MUST match Notebook A exactly.

In [7]:
from pynqsim import SimulationClient

SERVER_IP = "172.20.166.171"   # <-- same server IP as Notebook A
SERVER_PORT = 9003             # <-- same server port as Notebook A

blue = SimulationClient(SERVER_IP, SERVER_PORT)
print("Team Blue connected to server!")

Team Blue connected to server!


In [8]:
blue.join_competition(team_id="team_blue")
print("Joined as team_blue!")

Joined as team_blue!


In [9]:
state = blue.get_competition_state()
print(f"Current turn: {state.get('current_turn')}")
print(f"Scores: {state.get('scores', {})}")

Current turn: team_blue
Scores: {'team_red': 13, 'team_blue': 0}


## Color names (6x5 grid → 15 colors)
MUST match the order the scene assigns `color_idx` (indices 0..14).

In [10]:
# 15 colors, in the exact order the scene assigns color_idx (0..14).
COLOR_NAMES = [
    "RED", "GRN", "BLU", "YEL", "ORG",
    "PUR", "BRN", "PNK", "CYN", "MAG",
    "LIM", "TEA", "GRY", "MAR", "NVY",
]

def color_name(ci):
    return COLOR_NAMES[ci] if (ci is not None and ci < len(COLOR_NAMES)) else str(ci)

## Normal play (`flip`)
Drives the BLUE arm. Wait until it's Blue's turn or the server raises "Not your
turn". Match = go again; mismatch = re-cover + turn passes.

In [12]:
def flip(client, row, col):
    """Flip a card WITH memory-game scoring/turn messaging."""
    result = client.flip_card(row, col)
    if result.get("already_flipped"):
        print(f"Card ({row},{col}) already flipped")
        return result
    print(f">>> ({row},{col}) revealed: {color_name(result['color_idx'])}")
    mr = result.get("match_result")
    if mr:
        if mr.get("matched"):
            print("*** MATCH! +1 — same team goes again! ***")
        elif mr.get("turn_ended"):
            print("No match. Cards re-covered. Turn passes to the other team.")
    return result

flip(blue, 3, 3)   # first card
flip(blue, 5, 1) # second card

>>> (3,3) revealed: MAR
>>> (5,1) revealed: MAR
*** MATCH! +1 — same team goes again! ***


{'color_idx': 13,
 'already_flipped': False,
 'matched': False,
 'match_result': {'cards_flipped': 2,
  'matched': True,
  'turn_ended': False,
  'same_team_again': True},
 'status': 'ok'}

## `flip_raw()` — grab a cover with NO scoring / turn / re-cover
Calls the **`flip_raw` server action** (add the server snippet first). Skips
`record_flip`, so no scorekeeping, no turn check, and mismatched covers are NOT
teleported back — the cover stays in the discard pile. `robot_id` picks the arm:
**0=red (left cols), 1=blue (right cols)**, since one arm can't reach all 30 cards.

In [ ]:
def flip_raw(client, row, col, robot_id=None):
    """Grab one cover: no scoring, no turn logic, no re-cover."""
    params = {"row": row, "col": col}
    if robot_id is not None:
        params["robot_id"] = robot_id
    result = client._request("flip_raw", params)
    if result.get("already_flipped"):
        print(f"({row},{col}) already flipped")
    else:
        print(f"grabbed ({row},{col}) -> {color_name(result.get('color_idx'))}")
    return result

## Loop: clear the whole board
RED arm (0) takes the left columns, BLUE arm (1) the right columns, so both stay
in reach. Reads the true grid size from the server.

In [ ]:
def clear_board(client):
    grid = client.get_grid_state()
    n_rows = len(grid)
    n_cols = len(grid[0]) if n_rows else 0
    mid = n_cols // 2
    print(f"Clearing {n_rows}x{n_cols} board (red=left cols, blue=right cols)...")
    for r in range(n_rows):
        for c in range(n_cols):
            arm = 0 if c < mid else 1   # 0=red (left half), 1=blue (right half)
            try:
                flip_raw(client, r, c, robot_id=arm)
            except Exception as e:
                print(f"({r},{c}) failed: {e}")
    print("Board cleared.")

# clear_board(blue)

## Cleanup

In [ ]:
blue.leave_competition()
print("Team Blue left.")